In [ ]:
#The purpose of this script is to collect labelled data for 3 classifiers:
#1) An LSTM to recognise when a strikefoot is happening
#2) A Random Forest classifier to determine the strikefoot type (heel, midfoot, forefoot)
#3) A classifier to determine whether overstriding is occurring

#This script combines the functionality of strikefoot_type_trainer and data_extraction_1_2_3.
#A key limitation of data_extraction_1_2_3 was that the "u" frames were not saved. This fixes that.
import cv2
import numpy as np
import json
import sys, os
import pandas as pd
import torch
sys.path.insert(0, "/Users/abhinavarora/Desktop/CadenceCV/ml/src")
from utils.numpy_encoder import NumpyEncoder
from utils.video_extracter import extract_keypoints
from training.data_loader import training_df, testing_df
from torch import nn
from scipy.stats import zscore
from collections import deque
from torch.utils.data import Dataset, DataLoader

In [ ]:
def _tibial(shin_vec_arr, direction=1):
    # direction flips the shin x-component so the signed lean is measured relative to travel
    # direction, not the raw image x-axis. direction=1 leaves it unchanged (rightward runner).
    return np.pi / 2 - np.arctan2(shin_vec_arr[:, 1], direction * shin_vec_arr[:, 0])

def draw_ankle_line(frame, actual_frame_idx, which_leg, left_ankle_arr, right_ankle_arr):
    idx = actual_frame_idx
    ankle = left_ankle_arr[idx] if which_leg == 'left' else right_ankle_arr[idx]
    ax, ay = int(ankle[0]), int(ankle[1])
    if ax > 0:
        cv2.line(frame, (ax, ay), (ax, 0), (0, 0, 255), 2)
    return frame


In [ ]:
extensions = [".mp4", ".MOV"]
base_video_dir = ""

with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/frame_by_frame_data.json", "r") as file:
    data = json.load(file)

videos = set(frame["video"] for frame in data)

all_res = []

In [ ]:
#Update per video
def compute_all_metrics(actual_frame, which_leg, coords, video_name):

    left_ankle_arr = coords['left_ankle'];    right_ankle_arr = coords['right_ankle']
    left_hip_arr   = coords['left_hip'];      right_hip_arr   = coords['right_hip']
    left_knee_arr  = coords['left_knee'];     right_knee_arr  = coords['right_knee']
    left_shoulder_arr  = coords['left_shoulder']; right_shoulder_arr = coords['right_shoulder']
    total_frames = coords['Frame Count']
    n_detected   = len(left_ankle_arr)
    print(f'Keypoints detected in {n_detected}/{total_frames} frames')

    # Runner travel direction from forward trunk lean: the shoulder sits ahead of the hip in the
    # direction of travel. The median keeps it robust on a treadmill, where net hip displacement
    # is ~0 and a start-vs-end hip comparison would just be noise.
    # direction = +1 rightward (+X), -1 leftward (-X). Matches obtain_metrics.compute_all_metrics.
    mean_shoulder_x = (left_shoulder_arr[:, 0] + right_shoulder_arr[:, 0]) / 2
    mean_hip_x = (left_hip_arr[:, 0] + right_hip_arr[:, 0]) / 2
    direction = 1 if np.median(mean_shoulder_x - mean_hip_x) > 0 else -1

    # Horizontal signed features mirror-flip with running direction, so each is multiplied by
    # `direction` to canonicalise every runner to rightward travel. ankle_y_vel is vertical
    # (direction-independent) and is left uncorrected.
    left_ankle_x_vel = direction * np.gradient(left_ankle_arr[:, 0])
    left_ankle_y_vel = np.gradient(left_ankle_arr[:, 1])
    right_ankle_x_vel = direction * np.gradient(right_ankle_arr[:, 0])
    right_ankle_y_vel = np.gradient(right_ankle_arr[:, 1])

    left_shin_vec = left_knee_arr - left_ankle_arr
    right_shin_vec = right_knee_arr - right_ankle_arr
    left_shin_velocity  = direction * np.gradient(left_shin_vec[:, 0])
    right_shin_velocity = direction * np.gradient(right_shin_vec[:, 0])
    left_tibial_angle  = _tibial(left_shin_vec, direction)
    right_tibial_angle = _tibial(right_shin_vec, direction)

    # Signed inter-ankle distance — positive when THIS foot is ahead in direction of travel
    left_ankle_x_dist  = direction * (left_ankle_arr[:, 0] - right_ankle_arr[:, 0])
    right_ankle_x_dist = direction * (right_ankle_arr[:, 0] - left_ankle_arr[:, 0])
    idx = min(actual_frame, n_detected - 1)
    if which_leg == 'left':
        ax_vel = left_ankle_x_vel[idx];  ay_vel = left_ankle_y_vel[idx]
        sh_vel = left_shin_velocity[idx]; tib    = left_tibial_angle[idx]
        dist   = left_ankle_x_dist[idx]
        ankle_arr = left_ankle_arr;  knee_arr  = left_knee_arr
        hip_arr   = left_hip_arr;    shldr_arr = left_shoulder_arr
        all_ay    = left_ankle_y_vel
    else:
        ax_vel = right_ankle_x_vel[idx]; ay_vel = right_ankle_y_vel[idx]
        sh_vel = right_shin_velocity[idx];tib    = right_tibial_angle[idx]
        dist   = right_ankle_x_dist[idx]
        ankle_arr = right_ankle_arr; knee_arr  = right_knee_arr
        hip_arr   = right_hip_arr;   shldr_arr = right_shoulder_arr
        all_ay    = right_ankle_y_vel

    v1 = hip_arr[idx] - knee_arr[idx]
    v2 = ankle_arr[idx] - knee_arr[idx]
    knee_flexion = float(np.rad2deg(np.arctan2(v2[1], v2[0]) - np.arctan2(v1[1], v1[0])) % 360)
    offset = float(np.abs(ankle_arr[idx, 0] - hip_arr[idx, 0]))
    tv = shldr_arr[idx] - hip_arr[idx]
    trunk_angle = float(np.rad2deg(np.pi / 2 - np.arctan2(tv[1], tv[0])))
    pre2 = max(0, idx - 2)
    ankle_vel_pre = float(all_ay[pre2])
    p1 = max(0, idx - 1);  p3 = max(0, idx - 3)
    dx = float(ankle_arr[p1, 0] - ankle_arr[p3, 0])
    dy = float(ankle_arr[p1, 1] - ankle_arr[p3, 1])
    approach_angle = float(np.arctan2(dy, dx))

    return {
        'ankle_x_vel': float(ax_vel),
        'ankle_y_vel': float(ay_vel),
        'shin_velocity': float(sh_vel),
        'tibial_angle': float(tib),
        'ankle_x_dist': float(dist),
        'knee_flexion_angle': knee_flexion,
        'ankle_hip_horizontal_offset': offset,
        'trunk_angle': trunk_angle,
        'ankle_velocity_pre_strike': ankle_vel_pre,
        'ankle_approach_angle': approach_angle,
        'frame': idx,
        'side': which_leg,
        'video': video_name
    }

In [ ]:
for v in videos:
    for ext in extensions:
        try:
            coords = extract_keypoints(os.path.join(base_video_dir, v + ext))
            frames = coords["Frames"]
            for f in frames:
                for leg in ["left", "right"]:
                    res = compute_all_metrics(f, leg, coords, v)
                    all_res.append(res)
            
            break
        except:
            continue

In [ ]:
with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/frame_by_frame_data.json", "w") as file:
    json.dump(all_res, file, indent=4, cls=NumpyEncoder)

In [ ]:
#This function will go through the video frame by frame. Each frame is classified as:
#"u" = the EXACT first frame where the foot begins going up (toe-off)
#"d" = the EXACT first frame where the foot contacts the ground (strikefoot)
#"s" = skip (if the frame is neither u nor d)
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively.
#Each consecutive pair of "u" and "d" will be classified as a gait cycle.
def scrub_and_label(video_dir, which_leg):
    side_char = 'L' if which_leg == 'left' else 'R'
    cap = cv2.VideoCapture(video_dir)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    raw_labels = []
    current_u_frame = None

    for frame_idx in range(total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame = draw_ankle_line(frame, frame_idx, which_leg)

        #Helps in identifying which foot is to be tracked
        h, w = frame.shape[:2]
        #Scale font size relative to frame width (1280 used as a reference baseline)
        scale = w / 1280
        fs = scale * 0.55
        fs_bottom = scale * 0.9
        th = max(1, int(scale * 2))
        mg = int(scale * 15)
        pink = (203, 192, 255)

        cv2.putText(frame, f'{which_leg.upper()} | Frame {frame_idx}',
                    (mg, mg + 24), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 0), th, cv2.LINE_AA)
        if current_u_frame is not None:
            cv2.putText(frame, f'u_frame={current_u_frame}',
                        (mg, mg + 50), cv2.FONT_HERSHEY_SIMPLEX, fs, (200, 200, 0), th, cv2.LINE_AA)
        cv2.putText(frame, 'u=up  d=down  s=skip  c=quit',
                    (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)

        cv2.imshow('Labelling', frame)
        key = cv2.waitKey(0) & 0xFF

        #Constructing a single gait cycle — u_frame is stored until the matching d completes it
        if key == ord('u'):
            current_u_frame = frame_idx

        elif key == ord('d'):
            #"d" is always the strikefoot frame — immediately prompt for strike type then overstriding
            strike_frame = frame.copy()
            #Input classification (h = heel, m = midfoot, f = forefoot, s = skip this strike entirely)
            cv2.putText(strike_frame, 'Strike: h=heel  m=midfoot  f=forefoot  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', strike_frame)
            pk = cv2.waitKey(0) & 0xFF

            if pk == ord('s'):
                current_u_frame = None
                continue

            strike_map = {ord('h'): 'heel', ord('m'): 'midfoot', ord('f'): 'forefoot'}
            if pk not in strike_map:
                current_u_frame = None
                continue
            strike_pattern = strike_map[pk]

            #Input overstriding classification — the vertical ankle line helps judge this
            #If the line falls anterior to the pelvis it is overstriding, if within it is neutral
            over_frame = frame.copy()
            cv2.putText(over_frame, 'Overstride: o=yes  n=neutral  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', over_frame)
            ok_key = cv2.waitKey(0) & 0xFF

            overstride_map = {ord('o'): 'overstriding', ord('n'): 'neutral', ord('s'): 'skip'}
            overstriding = overstride_map.get(ok_key, 'skip')

            #Appending the completed gait cycle and resetting u_frame for the next cycle
            raw_labels.append({
                'frame': frame_idx,
                'side': side_char,
                'strike_pattern': strike_pattern,
                'overstriding': overstriding,
                'u_frame': current_u_frame,
            })
            current_u_frame = None

        elif key == ord('c'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return raw_labels

#These metrics must be extracted for ALL frames in each video before scrubbing begins
#Manual labelling done via tracking the left foot first
left_raw  = scrub_and_label(video_dir, 'left')
#This time, the right foot will be focused on to identify its gait cycles
right_raw = scrub_and_label(video_dir, 'right')

In [ ]:
#Obtaining metrics for each individual strikefoot frame and combining with the raw labels
#left and right entries are merged into a single list and sorted by frame number
all_entries = []

for raw in left_raw + right_raw:
    which_leg = 'left' if raw['side'] == 'L' else 'right'
    metrics = compute_all_metrics(raw['frame'], which_leg)
    entry = {
        'frame': raw['frame'],
        'side': raw['side'],
        'strike_pattern': raw['strike_pattern'],
        'overstriding': raw['overstriding'],
        'u_frame': raw['u_frame'],
        'video': video_name,
        **metrics,
    }
    all_entries.append(entry)

all_entries.sort(key=lambda x: (x['frame'], x['side']))
print(f'Total labeled strikes: {len(all_entries)}')

In [ ]:
#Data loading loop:
#1) Read existing data from strikefoot_data.json
#2) To the same list, add the new entries for this video
#3) Finally, rewrite strikefoot_data.json with the combined data
#If the file doesn't exist or holds old-format data (a dict), start fresh as an empty list
json_path = "/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_data.json"
with open(json_path, 'r') as f:
    existing = json.load(f)

existing.extend(all_entries)
with open(json_path, 'w') as f:
    json.dump(existing, f, cls=NumpyEncoder, indent=4)

In [ ]:
from utils.obtain_metrics import compute_all_metrics as run_yolo_metrics

extensions = ['.mp4', '.MOV']
base_video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
strikefoot_json_path = "/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_data.json"

with open(strikefoot_json_path, 'r') as f:
    sf_data = json.load(f)

video_names = set()
for entry in sf_data:
    video_names.add(entry['video'])

all_frame_by_frame_data = {}

for vname in video_names:
    for ext in extensions:
        try:
            video_path = os.path.join(base_video_dir, vname + ext)
            results, frame_count, duration, *_ = run_yolo_metrics(video_path)
            all_frame_by_frame_data[vname] = results
            print(f"Done: {vname}{ext} — {len(results)} frame entries")
            break
        except Exception:
            continue

In [ ]:
frame_by_frame_json_path = "/Users/abhinavarora/Desktop/CadenceCV/ml/data/frame_by_frame_data.json"
with open(frame_by_frame_json_path, 'w') as f:
    json.dump(all_frame_by_frame_data, f, indent=4, cls=NumpyEncoder)
print(f"Written {len(all_frame_by_frame_data)} videos to frame_by_frame_data.json")

In [ ]:
video_extensions = [".mp4", ".MOV"]
base_video_dir_path = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
vids = []
with open (json_path, "r") as file:
    data = json.load(file)

for frame in data:
    vids.append(frame["video"])


metadata = []
for video in vids:
    for ext in video_extensions:
        try:
            cap = cv2.VideoCapture(base_video_dir_path + "/" + video + ext)
            fps = cap.get(cv2.CAP_PROP_POS_FRAMES)
            width  = cap.get(cv2.CV_CAP_PROP_FRAME_WIDTH)
            height = cap.get(cv2.CV_CAP_PROP_FRAME_HEIGHT)
            print(width, height, fps)
            metadata.append({"Video": video, "Width": width, "Height": height, "FPS": fps})
            break
        except:
            pass

with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/video_meta_data.json", "w") as file:
    json.dump(metadata, file, indent=4, cls=NumpyEncoder)